In [1]:
from __future__ import annotations

import argparse
import hashlib
import json
import math
import random
import re
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
from tqdm import tqdm

In [2]:
REQUIRED_COLUMNS = [
    "article_id",
    "chunk_id",
    "text",
    "label",
    "ai_spans_json",
    "ai_fraction",
    "has_mixed_content",
    "strategy",
    "source_model",
    "judge_semantic_score",
    "judge_complexity_score",
    "judge_avg",
    "word_count",
    "char_count",
    "chunk_index",
    "total_chunks",
    "original_chunk_id",
    "created_at",
]

LABEL_MAP = {
    0: "human",
    1: "full_ai",
    2: "mixed",
}

REFUSAL_PATTERNS = [
    "я не могу помочь",
    "я не могу выполнить",
    "я не могу предоставить",
    "извините, но я не могу",
    "не могу помочь с этим",
    "i can't help with that",
    "i cannot help with that",
    "i can't assist with that",
    "i’m sorry, but i can’t",
    "i am sorry, but i can't",
]

PROMPT_LEAK_PATTERNS = [
    "вот переработанный фрагмент",
    "вот продолжение",
    "output only",
    "translate the following",
    "только текст фрагмента",
    "фрагмент:",
    "text:",
    "продолжай отсюда",
    "you are a professional translator",
    "ты редактор",
    "ты — редактор",
    "ты соавтор",
]

TOPIC_KEYWORDS = {
    "religion": [
        "религ", "церков", "христиан", "ислам", "будд", "православ",
        "мусульман", "коран", "библи", "священ", "вера", "богослов", "бог"
    ],
    "politics": [
        "полит", "государств", "власть", "парти", "выбор", "депутат",
        "президент", "конституц", "санкц", "идеолог", "режим", "геополит"
    ],
    "philosophy": [
        "философ", "онтолог", "эпистем", "гносеолог", "метафиз",
        "экзистен", "сознани", "бытие", "этика", "аксиолог", "герменевт"
    ],
}

In [3]:
def ensure_output_dirs(base_dir: Path) -> Dict[str, Path]:
    """
    Создаёт структуру директорий для артефактов.
    """
    paths = {
        "base": base_dir,
        "processed": base_dir / "processed",
        "reports": base_dir / "reports",
    }
    for p in paths.values():
        p.mkdir(parents=True, exist_ok=True)
    return paths


def safe_str(value: Any) -> str:
    """
    Приводит значение к строке без изменения внутреннего содержимого текста.
    Важно: whitespace внутри текста НЕ нормализуем, чтобы не сломать char offsets.
    """
    if pd.isna(value):
        return ""
    return str(value)


def safe_int(value: Any) -> Optional[int]:
    """
    Надёжное преобразование nullable значений в int.
    """
    if pd.isna(value):
        return None
    try:
        return int(value)
    except Exception:
        return None


def safe_float(value: Any) -> Optional[float]:
    """
    Надёжное преобразование nullable значений в float.
    """
    if pd.isna(value):
        return None
    try:
        return float(value)
    except Exception:
        return None


def count_words(text: str) -> int:
    """
    Простая оценка количества слов.
    Используем токены по пробелам, этого достаточно для EDA.
    """
    return len(re.findall(r"\S+", text))


def hash_text(text: str) -> str:
    """
    Хэш текста для поиска exact duplicates.
    """
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def is_nan_or_empty(value: Any) -> bool:
    """
    Проверка на пустое значение.
    """
    if pd.isna(value):
        return True
    if isinstance(value, str) and value.strip() == "":
        return True
    return False


def try_parse_json_spans(raw_value: Any) -> Tuple[Optional[List[List[int]]], Optional[str]]:
    """
    Парсит ai_spans_json.
    Ожидаемый формат: [[start, end], [start2, end2], ...]
    """
    if pd.isna(raw_value):
        return [], None

    raw_str = str(raw_value).strip()
    if raw_str == "":
        return [], None

    try:
        parsed = json.loads(raw_str)
    except Exception as e:
        return None, f"invalid_ai_spans_json: {e}"

    if not isinstance(parsed, list):
        return None, "ai_spans_json_not_a_list"

    normalized: List[List[int]] = []
    for item in parsed:
        if not isinstance(item, list) or len(item) != 2:
            return None, "ai_spans_json_item_not_pair"
        start, end = item
        try:
            start = int(start)
            end = int(end)
        except Exception:
            return None, "ai_spans_json_non_int_bounds"
        normalized.append([start, end])

    return normalized, None


def normalize_ai_spans(ai_spans: List[List[int]], text_len: int) -> Tuple[Optional[List[List[int]]], List[str]]:
    """
    Валидирует и нормализует AI spans:
    - сортировка
    - проверка границ
    - слияние касающихся span-ов
    - слияние overlap span-ов с флагом
    """
    flags: List[str] = []

    if ai_spans is None:
        return None, ["ai_spans_is_none"]

    for start, end in ai_spans:
        if start < 0 or end < 0:
            return None, ["negative_ai_span_bound"]
        if end <= start:
            return None, ["zero_or_negative_length_ai_span"]
        if end > text_len:
            return None, ["ai_span_out_of_bounds"]

    spans_sorted = sorted(ai_spans, key=lambda x: (x[0], x[1]))
    merged: List[List[int]] = []

    for start, end in spans_sorted:
        if not merged:
            merged.append([start, end])
            continue

        last_start, last_end = merged[-1]

        if start < last_end:
            # overlap — это плохо, но лучше слить и зафлаговать, чем silently поломать датасет
            merged[-1][1] = max(last_end, end)
            flags.append("merged_overlapping_ai_spans")
        elif start == last_end:
            # касаются — можно безопасно слить
            merged[-1][1] = end
            flags.append("merged_adjacent_ai_spans")
        else:
            merged.append([start, end])

    return merged, flags


def build_canonical_spans(text_len: int, ai_spans: List[List[int]]) -> List[Dict[str, Any]]:
    """
    Преобразует список AI span-ов в полное покрытие текста:
    human + ai spans.
    """
    result: List[Dict[str, Any]] = []
    cursor = 0

    for start, end in ai_spans:
        if start > cursor:
            result.append({"start": cursor, "end": start, "label": "human"})
        result.append({"start": start, "end": end, "label": "ai"})
        cursor = end

    if cursor < text_len:
        result.append({"start": cursor, "end": text_len, "label": "human"})

    if text_len == 0:
        return []

    # Если AI span-ов нет — весь текст human
    if not result:
        result = [{"start": 0, "end": text_len, "label": "human"}]

    return result


def compute_ai_char_fraction(ai_spans: List[List[int]], text_len: int) -> float:
    """
    Вычисляет долю AI символов по span-разметке.
    """
    if text_len == 0:
        return 0.0
    ai_chars = sum(end - start for start, end in ai_spans)
    return ai_chars / text_len


def detect_refusal(text: str) -> bool:
    """
    Грубая эвристика для детекта refusal-ответов модели.
    """
    lower = text.lower()
    return any(pattern in lower for pattern in REFUSAL_PATTERNS)


def detect_prompt_leak(text: str) -> bool:
    """
    Грубая эвристика для утечки инструкций/промпта в текст.
    """
    lower = text.lower()
    return any(pattern in lower for pattern in PROMPT_LEAK_PATTERNS)


def detect_topic_flags(text: str) -> List[str]:
    """
    Keyword-based флаггер нежелательных тем.
    Это не истина, а только pre-filter на ручную проверку.
    """
    lower = text.lower()
    matched_topics = []

    for topic, keywords in TOPIC_KEYWORDS.items():
        if any(keyword in lower for keyword in keywords):
            matched_topics.append(topic)

    return matched_topics


def compute_stats_numeric(values: List[float]) -> Dict[str, Optional[float]]:
    """
    Универсальная агрегированная статистика для числовых списков.
    """
    clean_values = [float(v) for v in values if v is not None and not math.isnan(v)]
    if not clean_values:
        return {
            "count": 0,
            "min": None,
            "mean": None,
            "median": None,
            "max": None,
            "std": None,
        }

    arr = np.array(clean_values, dtype=float)
    return {
        "count": int(arr.size),
        "min": round(float(np.min(arr)), 6),
        "mean": round(float(np.mean(arr)), 6),
        "median": round(float(np.median(arr)), 6),
        "max": round(float(np.max(arr)), 6),
        "std": round(float(np.std(arr)), 6),
    }


def assign_group_splits(
    records: List[Dict[str, Any]],
    val_size: float,
    test_size: float,
    seed: int
) -> Dict[str, str]:
    """
    Делит данные по group_id, чтобы не было leakage между сплитами.
    Возвращает mapping: group_id -> split
    """
    group_ids = sorted({str(r["group_id"]) for r in records})
    rng = random.Random(seed)
    rng.shuffle(group_ids)

    n = len(group_ids)
    n_test = int(round(n * test_size))
    n_val = int(round(n * val_size))

    # На маленьких датасетах гарантируем, что train не опустеет
    if n >= 3:
        if n_test == 0:
            n_test = 1
        if n_val == 0:
            n_val = 1
        if n_test + n_val >= n:
            n_val = max(1, n_val - 1)

    test_groups = set(group_ids[:n_test])
    val_groups = set(group_ids[n_test:n_test + n_val])
    train_groups = set(group_ids[n_test + n_val:])

    split_map: Dict[str, str] = {}
    for gid in train_groups:
        split_map[gid] = "train"
    for gid in val_groups:
        split_map[gid] = "val"
    for gid in test_groups:
        split_map[gid] = "test"

    return split_map


def write_json(path: Path, payload: Dict[str, Any]) -> None:
    """
    Сохраняет JSON.
    """
    with path.open("w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)


def write_jsonl(path: Path, records: List[Dict[str, Any]]) -> None:
    """
    Сохраняет JSONL.
    """
    with path.open("w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

In [4]:
def validate_required_columns(df: pd.DataFrame) -> None:
    """
    Проверяет наличие обязательных колонок.
    """
    missing = [col for col in REQUIRED_COLUMNS if col not in df.columns]
    if missing:
        raise ValueError(f"В CSV не хватает обязательных колонок: {missing}")


def build_record_from_row(
    row: pd.Series,
    low_judge_threshold: float
) -> Tuple[Optional[Dict[str, Any]], Dict[str, Any]]:
    """
    Преобразует одну строку CSV в канонический record + issue entry.
    Если запись критически плохая — возвращает record=None и статус drop.
    """
    issues: List[str] = []
    fatal_issues: List[str] = []

    text = safe_str(row.get("text"))
    if text == "":
        fatal_issues.append("empty_text")

    text_len = len(text)
    actual_word_count = count_words(text)
    actual_char_count = len(text)

    chunk_id = safe_str(row.get("chunk_id"))
    article_id = safe_str(row.get("article_id"))
    original_chunk_id = safe_str(row.get("original_chunk_id")) or chunk_id

    label = safe_int(row.get("label"))
    if label not in (0, 1, 2):
        fatal_issues.append("invalid_label")

    # reported counts
    reported_word_count = safe_int(row.get("word_count"))
    reported_char_count = safe_int(row.get("char_count"))
    reported_ai_fraction = safe_float(row.get("ai_fraction"))

    if reported_char_count is not None and reported_char_count != actual_char_count:
        issues.append("reported_char_count_mismatch")

    if reported_word_count is not None and abs(reported_word_count - actual_word_count) > 3:
        issues.append("reported_word_count_mismatch")

    # parse ai_spans_json
    parsed_ai_spans, parse_error = try_parse_json_spans(row.get("ai_spans_json"))
    if parse_error is not None:
        fatal_issues.append(parse_error)

    # derive ai spans from label + parsed json
    ai_spans: Optional[List[List[int]]] = None
    if label == 0:
        ai_spans = []
        if parsed_ai_spans not in ([], None):
            issues.append("label_0_but_ai_spans_present")
    elif label == 1:
        if text_len == 0:
            fatal_issues.append("label_1_with_empty_text")
        ai_spans = [[0, text_len]]
        if parsed_ai_spans not in ([], [[0, text_len]], None):
            issues.append("label_1_but_ai_spans_not_full_text")
    elif label == 2:
        if parsed_ai_spans is None:
            fatal_issues.append("label_2_but_ai_spans_unavailable")
        else:
            if len(parsed_ai_spans) == 0:
                # fallback на start/end char, если json пустой
                start_char = safe_int(row.get("ai_span_start_char"))
                end_char = safe_int(row.get("ai_span_end_char"))
                if start_char is not None and end_char is not None:
                    ai_spans = [[start_char, end_char]]
                    issues.append("label_2_ai_span_restored_from_start_end_char")
                else:
                    fatal_issues.append("label_2_without_ai_spans")
            else:
                ai_spans = parsed_ai_spans

    # normalize spans
    if ai_spans is not None:
        ai_spans, span_flags = normalize_ai_spans(ai_spans, text_len)
        if ai_spans is None:
            fatal_issues.extend(span_flags)
        else:
            issues.extend(span_flags)

    if not fatal_issues and ai_spans is None:
        fatal_issues.append("ai_spans_resolution_failed")

    # derived fraction vs reported fraction
    derived_ai_fraction = None
    if ai_spans is not None:
        derived_ai_fraction = compute_ai_char_fraction(ai_spans, text_len)
        if reported_ai_fraction is not None and abs(reported_ai_fraction - derived_ai_fraction) > 0.08:
            issues.append("ai_fraction_mismatch")

    # consistency flags
    has_mixed_content = row.get("has_mixed_content")
    if label == 2 and str(has_mixed_content).lower() not in ("true", "1"):
        issues.append("label_2_but_has_mixed_content_not_true")

    if label in (0, 1) and str(has_mixed_content).lower() in ("true", "1"):
        issues.append("non_mixed_label_but_has_mixed_content_true")

    # quality heuristics
    if detect_refusal(text):
        fatal_issues.append("suspected_refusal_text")

    if detect_prompt_leak(text):
        issues.append("prompt_leak_pattern")

    topic_flags = detect_topic_flags(text)
    for topic in topic_flags:
        issues.append(f"topic_flag_{topic}")

    strategy = safe_str(row.get("strategy"))
    source_model = safe_str(row.get("source_model"))

    judge_semantic_score = safe_float(row.get("judge_semantic_score"))
    judge_complexity_score = safe_float(row.get("judge_complexity_score"))
    judge_avg = safe_float(row.get("judge_avg"))

    is_synthetic = strategy != "original" or source_model.lower() != "human" or label in (1, 2)

    if is_synthetic and judge_avg is None:
        issues.append("missing_judge_avg_for_synthetic")

    if is_synthetic and judge_avg is not None and judge_avg < low_judge_threshold:
        issues.append("low_judge_avg")

    if actual_char_count < 50:
        issues.append("too_short_text")

    if fatal_issues:
        issue_entry = {
            "chunk_id": chunk_id,
            "article_id": article_id,
            "status": "drop",
            "fatal_issues": fatal_issues,
            "issues": issues,
        }
        return None, issue_entry

    canonical_spans = build_canonical_spans(text_len, ai_spans)
    ai_char_count = sum(end - start for start, end in ai_spans)

    # status policy
    quarantine_flags = set()

    for issue in issues:
        if issue.startswith("topic_flag_"):
            quarantine_flags.add(issue)

    soft_quarantine_markers = {
        "low_judge_avg",
        "prompt_leak_pattern",
        "missing_judge_avg_for_synthetic",
        "reported_char_count_mismatch",
        "reported_word_count_mismatch",
        "ai_fraction_mismatch",
        "too_short_text",
    }
    for issue in issues:
        if issue in soft_quarantine_markers:
            quarantine_flags.add(issue)

    status = "quarantine" if len(quarantine_flags) > 0 else "clean"

    record = {
        "id": chunk_id,
        "group_id": article_id if article_id != "" else original_chunk_id,
        "source_id": original_chunk_id,
        "text": text,
        "spans": canonical_spans,
        "meta": {
            "dataset_label": label,
            "dataset_label_name": LABEL_MAP[label],
            "strategy": strategy,
            "source_model": source_model,
            "ai_fraction_reported": reported_ai_fraction,
            "ai_fraction_derived": round(derived_ai_fraction, 6) if derived_ai_fraction is not None else None,
            "has_mixed_content": bool(str(has_mixed_content).lower() in ("true", "1")),
            "judge_semantic_score": judge_semantic_score,
            "judge_complexity_score": judge_complexity_score,
            "judge_avg": judge_avg,
            "word_count_reported": reported_word_count,
            "word_count_actual": actual_word_count,
            "char_count_reported": reported_char_count,
            "char_count_actual": actual_char_count,
            "article_id": row.get("article_id"),
            "chunk_index": safe_int(row.get("chunk_index")),
            "total_chunks": safe_int(row.get("total_chunks")),
            "original_chunk_id": original_chunk_id,
            "created_at": safe_str(row.get("created_at")),
            "ai_char_count": ai_char_count,
            "text_hash": hash_text(text),
        },
        "quality": {
            "status": status,
            "flags": sorted(list(set(issues))),
        },
        "split": None,
    }

    issue_entry = {
        "chunk_id": chunk_id,
        "article_id": article_id,
        "status": status,
        "fatal_issues": [],
        "issues": sorted(list(set(issues))),
    }
    return record, issue_entry


def add_duplicate_flags(records: List[Dict[str, Any]], issues_table: List[Dict[str, Any]]) -> None:
    """
    Добавляет флаги exact duplicate text.
    """
    hash_to_ids = defaultdict(list)
    id_to_issue = {row["chunk_id"]: row for row in issues_table}

    for record in records:
        text_hash = record["meta"]["text_hash"]
        hash_to_ids[text_hash].append(record["id"])

    duplicate_hashes = {h: ids for h, ids in hash_to_ids.items() if len(ids) > 1}

    for record in records:
        text_hash = record["meta"]["text_hash"]
        if text_hash in duplicate_hashes:
            if "exact_duplicate_text" not in record["quality"]["flags"]:
                record["quality"]["flags"].append("exact_duplicate_text")

            if record["quality"]["status"] == "clean":
                record["quality"]["status"] = "quarantine"

            issue_row = id_to_issue.get(record["id"])
            if issue_row is not None:
                if "exact_duplicate_text" not in issue_row["issues"]:
                    issue_row["issues"].append("exact_duplicate_text")
                if issue_row["status"] == "clean":
                    issue_row["status"] = "quarantine"


def make_split_summary(records: List[Dict[str, Any]]) -> Dict[str, Any]:
    """
    Считает сводку по split-ам.
    """
    result: Dict[str, Any] = {}
    for split in ("train", "val", "test"):
        subset = [r for r in records if r.get("split") == split]
        result[split] = {
            "rows": len(subset),
            "unique_groups": len({str(r["group_id"]) for r in subset}),
            "label_distribution": dict(Counter(r["meta"]["dataset_label_name"] for r in subset)),
            "strategy_distribution": dict(Counter(r["meta"]["strategy"] for r in subset)),
        }
    return result


def build_eda_summary(
    input_df: pd.DataFrame,
    records_all: List[Dict[str, Any]],
    issues_table: List[Dict[str, Any]],
    split_summary: Dict[str, Any]
) -> Dict[str, Any]:
    """
    Формирует итоговый EDA summary.
    """
    clean_records = [r for r in records_all if r["quality"]["status"] == "clean"]
    quarantine_records = [r for r in records_all if r["quality"]["status"] == "quarantine"]

    drop_rows = [row for row in issues_table if row["status"] == "drop"]

    def extract_lengths(recs: List[Dict[str, Any]]) -> Tuple[List[int], List[int], List[float], List[int]]:
        char_lengths = [r["meta"]["char_count_actual"] for r in recs]
        word_lengths = [r["meta"]["word_count_actual"] for r in recs]
        ai_fracs = [
            r["meta"]["ai_fraction_derived"] for r in recs
            if r["meta"]["ai_fraction_derived"] is not None
        ]
        ai_span_lengths = []
        for r in recs:
            for span in r["spans"]:
                if span["label"] == "ai":
                    ai_span_lengths.append(span["end"] - span["start"])
        return char_lengths, word_lengths, ai_fracs, ai_span_lengths

    def label_dist(recs: List[Dict[str, Any]]) -> Dict[str, int]:
        return dict(Counter(r["meta"]["dataset_label_name"] for r in recs))

    def strategy_dist(recs: List[Dict[str, Any]]) -> Dict[str, int]:
        return dict(Counter(r["meta"]["strategy"] for r in recs))

    def source_model_dist(recs: List[Dict[str, Any]]) -> Dict[str, int]:
        return dict(Counter(r["meta"]["source_model"] for r in recs))

    def issue_counter(rows: List[Dict[str, Any]]) -> Dict[str, int]:
        cnt = Counter()
        for row in rows:
            for issue in row["issues"]:
                cnt[issue] += 1
            for issue in row["fatal_issues"]:
                cnt[issue] += 1
        return dict(cnt)

    all_char_lengths, all_word_lengths, all_ai_fracs, all_ai_span_lengths = extract_lengths(records_all)
    clean_char_lengths, clean_word_lengths, clean_ai_fracs, clean_ai_span_lengths = extract_lengths(clean_records)

    # strategy coverage by original_chunk_id family
    family_ids = set()
    strategy_to_families = defaultdict(set)

    for r in records_all:
        family_id = str(r["source_id"])
        family_ids.add(family_id)
        strategy_to_families[r["meta"]["strategy"]].add(family_id)

    strategy_family_coverage = {}
    total_families = max(1, len(family_ids))
    for strategy, fams in strategy_to_families.items():
        strategy_family_coverage[strategy] = {
            "families_covered": len(fams),
            "coverage_ratio": round(len(fams) / total_families, 6),
        }

    # judge stats
    judge_semantic = [
        r["meta"]["judge_semantic_score"]
        for r in records_all
        if r["meta"]["judge_semantic_score"] is not None
    ]
    judge_complexity = [
        r["meta"]["judge_complexity_score"]
        for r in records_all
        if r["meta"]["judge_complexity_score"] is not None
    ]
    judge_avg = [
        r["meta"]["judge_avg"]
        for r in records_all
        if r["meta"]["judge_avg"] is not None
    ]

    # ai char ratio overall
    total_chars = sum(r["meta"]["char_count_actual"] for r in records_all)
    total_ai_chars = sum(r["meta"]["ai_char_count"] for r in records_all)
    total_chars_clean = sum(r["meta"]["char_count_actual"] for r in clean_records)
    total_ai_chars_clean = sum(r["meta"]["ai_char_count"] for r in clean_records)

    # duplicate stats
    hash_counts = Counter(r["meta"]["text_hash"] for r in records_all)
    duplicate_groups = sum(1 for _, c in hash_counts.items() if c > 1)
    duplicate_rows = sum(c for _, c in hash_counts.items() if c > 1)

    summary = {
        "dataset_size": {
            "input_rows_csv": int(len(input_df)),
            "normalized_records_total": int(len(records_all)),
            "clean_records": int(len(clean_records)),
            "quarantine_records": int(len(quarantine_records)),
            "dropped_rows": int(len(drop_rows)),
            "unique_article_ids_input": int(input_df["article_id"].nunique(dropna=True)),
            "unique_chunk_ids_input": int(input_df["chunk_id"].nunique(dropna=True)),
            "unique_original_chunk_ids_input": int(input_df["original_chunk_id"].nunique(dropna=True)),
        },
        "label_distribution": {
            "all": label_dist(records_all),
            "clean": label_dist(clean_records),
        },
        "strategy_distribution": {
            "all": strategy_dist(records_all),
            "clean": strategy_dist(clean_records),
        },
        "source_model_distribution": {
            "all": source_model_dist(records_all),
            "clean": source_model_dist(clean_records),
        },
        "length_stats": {
            "all_char_count": compute_stats_numeric(all_char_lengths),
            "all_word_count": compute_stats_numeric(all_word_lengths),
            "clean_char_count": compute_stats_numeric(clean_char_lengths),
            "clean_word_count": compute_stats_numeric(clean_word_lengths),
        },
        "ai_content_stats": {
            "all_ai_fraction_derived": compute_stats_numeric(all_ai_fracs),
            "clean_ai_fraction_derived": compute_stats_numeric(clean_ai_fracs),
            "overall_ai_char_ratio_all": round(total_ai_chars / total_chars, 6) if total_chars > 0 else 0.0,
            "overall_ai_char_ratio_clean": round(total_ai_chars_clean / total_chars_clean, 6) if total_chars_clean > 0 else 0.0,
        },
        "ai_span_stats": {
            "all_ai_span_length_chars": compute_stats_numeric(all_ai_span_lengths),
            "clean_ai_span_length_chars": compute_stats_numeric(clean_ai_span_lengths),
        },
        "judge_stats": {
            "judge_semantic_score": compute_stats_numeric(judge_semantic),
            "judge_complexity_score": compute_stats_numeric(judge_complexity),
            "judge_avg": compute_stats_numeric(judge_avg),
        },
        "quality_issues": issue_counter(issues_table),
        "duplicates": {
            "exact_duplicate_groups": int(duplicate_groups),
            "rows_in_duplicate_groups": int(duplicate_rows),
        },
        "strategy_family_coverage": strategy_family_coverage,
        "split_summary": split_summary,
    }

    return summary


def process_dataset(
    input_csv: Path,
    output_dir: Path,
    val_size: float,
    test_size: float,
    seed: int,
    low_judge_threshold: float,
) -> None:
    """
    Основной pipeline.
    """
    dirs = ensure_output_dirs(output_dir)

    print(f"Читаю CSV: {input_csv}")
    df = pd.read_csv(input_csv)

    validate_required_columns(df)

    records_all: List[Dict[str, Any]] = []
    issues_table: List[Dict[str, Any]] = []

    for _, row in tqdm(df.iterrows(), total=len(df), desc="Auditing rows"):
        record, issue_entry = build_record_from_row(
            row=row,
            low_judge_threshold=low_judge_threshold,
        )
        issues_table.append(issue_entry)
        if record is not None:
            records_all.append(record)

    # duplicate flags после сборки всех records
    add_duplicate_flags(records_all, issues_table)

    # split only non-drop records; train/val/test — только clean
    clean_records = [r for r in records_all if r["quality"]["status"] == "clean"]
    split_map = assign_group_splits(
        records=clean_records,
        val_size=val_size,
        test_size=test_size,
        seed=seed,
    )

    for record in clean_records:
        record["split"] = split_map[str(record["group_id"])]

    # quarantine records split не назначаем — они не идут в основной train
    for record in records_all:
        if record["quality"]["status"] != "clean":
            record["split"] = None

    train_records = [r for r in clean_records if r["split"] == "train"]
    val_records = [r for r in clean_records if r["split"] == "val"]
    test_records = [r for r in clean_records if r["split"] == "test"]

    split_summary = make_split_summary(clean_records)
    eda_summary = build_eda_summary(df, records_all, issues_table, split_summary)

    # Сохраняем артефакты
    processed_dir = dirs["processed"]
    reports_dir = dirs["reports"]

    write_jsonl(processed_dir / "all_records.jsonl", records_all)
    write_jsonl(processed_dir / "clean_records.jsonl", clean_records)
    write_jsonl(processed_dir / "train.jsonl", train_records)
    write_jsonl(processed_dir / "val.jsonl", val_records)
    write_jsonl(processed_dir / "test.jsonl", test_records)

    issues_df = pd.DataFrame(issues_table)
    issues_df.to_csv(reports_dir / "issues.csv", index=False, encoding="utf-8")

    write_json(reports_dir / "eda_summary.json", eda_summary)

    print("\n=== Готово ===")
    print(f"Все записи:         {processed_dir / 'all_records.jsonl'}")
    print(f"Clean записи:       {processed_dir / 'clean_records.jsonl'}")
    print(f"Train:              {processed_dir / 'train.jsonl'}")
    print(f"Val:                {processed_dir / 'val.jsonl'}")
    print(f"Test:               {processed_dir / 'test.jsonl'}")
    print(f"Issues CSV:         {reports_dir / 'issues.csv'}")
    print(f"EDA summary JSON:   {reports_dir / 'eda_summary.json'}")

    print("\nКраткая сводка:")
    print(json.dumps(eda_summary["dataset_size"], ensure_ascii=False, indent=2))
    print(json.dumps(eda_summary["label_distribution"], ensure_ascii=False, indent=2))
    print(json.dumps(eda_summary["split_summary"], ensure_ascii=False, indent=2))

In [6]:
process_dataset(
    input_csv=Path("final_dataset.csv"),
    output_dir=Path("/token_detector_v1/"),
    val_size=0.1,
    test_size=0.1,
    seed=42,
    low_judge_threshold=3.0,
)

Читаю CSV: final_dataset.csv


Auditing rows: 100%|██████████| 3460/3460 [00:02<00:00, 1358.81it/s]



=== Готово ===
Все записи:         /token_detector_v1/processed/all_records.jsonl
Clean записи:       /token_detector_v1/processed/clean_records.jsonl
Train:              /token_detector_v1/processed/train.jsonl
Val:                /token_detector_v1/processed/val.jsonl
Test:               /token_detector_v1/processed/test.jsonl
Issues CSV:         /token_detector_v1/reports/issues.csv
EDA summary JSON:   /token_detector_v1/reports/eda_summary.json

Краткая сводка:
{
  "input_rows_csv": 3460,
  "normalized_records_total": 3460,
  "clean_records": 613,
  "quarantine_records": 2847,
  "dropped_rows": 0,
  "unique_article_ids_input": 195,
  "unique_chunk_ids_input": 3296,
  "unique_original_chunk_ids_input": 825
}
{
  "all": {
    "human": 825,
    "full_ai": 1489,
    "mixed": 1146
  },
  "clean": {
    "human": 101,
    "mixed": 199,
    "full_ai": 313
  }
}
{
  "train": {
    "rows": 501,
    "unique_groups": 77,
    "label_distribution": {
      "human": 86,
      "mixed": 159,
     

In [7]:
# =========================
# CELL 1. Imports & utils
# =========================

from __future__ import annotations

import copy
import json
import math
import random
import re
from collections import Counter, defaultdict
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any, Dict, List, Tuple, Optional

import numpy as np
import pandas as pd


def read_jsonl(path: Path) -> List[Dict[str, Any]]:
    """
    Читает JSONL в список dict.
    """
    records = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records


def write_jsonl(path: Path, records: List[Dict[str, Any]]) -> None:
    """
    Пишет список dict в JSONL.
    """
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")


def write_json(path: Path, payload: Dict[str, Any]) -> None:
    """
    Пишет JSON.
    """
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)


def assign_group_splits(
    records: List[Dict[str, Any]],
    val_size: float,
    test_size: float,
    seed: int
) -> Dict[str, str]:
    """
    Делит записи по group_id, чтобы не было leakage между train/val/test.
    """
    group_ids = sorted({str(r["group_id"]) for r in records})
    rng = random.Random(seed)
    rng.shuffle(group_ids)

    n = len(group_ids)
    n_test = int(round(n * test_size))
    n_val = int(round(n * val_size))

    if n >= 3:
        if n_test == 0:
            n_test = 1
        if n_val == 0:
            n_val = 1
        if n_test + n_val >= n:
            n_val = max(1, n_val - 1)

    test_groups = set(group_ids[:n_test])
    val_groups = set(group_ids[n_test:n_test + n_val])
    train_groups = set(group_ids[n_test + n_val:])

    split_map = {}
    for gid in train_groups:
        split_map[gid] = "train"
    for gid in val_groups:
        split_map[gid] = "val"
    for gid in test_groups:
        split_map[gid] = "test"

    return split_map


def safe_float(value: Any) -> Optional[float]:
    """
    Надёжное приведение к float.
    """
    try:
        if value is None:
            return None
        if isinstance(value, float) and math.isnan(value):
            return None
        return float(value)
    except Exception:
        return None


def compute_numeric_stats(values: List[float]) -> Dict[str, Any]:
    """
    Агрегированные stats для числового списка.
    """
    clean = [float(v) for v in values if v is not None and not math.isnan(v)]
    if not clean:
        return {"count": 0, "min": None, "mean": None, "median": None, "max": None, "std": None}

    arr = np.array(clean, dtype=float)
    return {
        "count": int(arr.size),
        "min": round(float(np.min(arr)), 6),
        "mean": round(float(np.mean(arr)), 6),
        "median": round(float(np.median(arr)), 6),
        "max": round(float(np.max(arr)), 6),
        "std": round(float(np.std(arr)), 6),
    }


def flatten_flags(records: List[Dict[str, Any]]) -> pd.DataFrame:
    """
    Разворачивает flags в табличный вид.
    """
    rows = []
    for r in records:
        flags = r.get("quality", {}).get("flags", [])
        if not flags:
            rows.append({
                "id": r["id"],
                "group_id": r["group_id"],
                "status": r["quality"]["status"],
                "flag": "__no_flags__",
                "label_name": r["meta"].get("dataset_label_name"),
                "strategy": r["meta"].get("strategy"),
                "judge_avg": r["meta"].get("judge_avg"),
            })
        else:
            for flag in flags:
                rows.append({
                    "id": r["id"],
                    "group_id": r["group_id"],
                    "status": r["quality"]["status"],
                    "flag": flag,
                    "label_name": r["meta"].get("dataset_label_name"),
                    "strategy": r["meta"].get("strategy"),
                    "judge_avg": r["meta"].get("judge_avg"),
                })
    return pd.DataFrame(rows)


In [8]:
# =========================
# CELL 2. Review policy
# =========================

@dataclass
class ReviewPolicy:
    """
    Политика пересмотра quarantine.
    Можно гибко менять, не переписывая логику.
    """

    # Если проблема только в служебных полях - не карантиним
    ignore_metadata_mismatches: bool = True

    # topic_flag_* не используем как причину quarantine
    allow_topic_flags_for_training: bool = True

    # ai_fraction mismatch не карантиним, если spans валидны
    allow_ai_fraction_mismatch: bool = True

    # отсутствие judge score само по себе не карантин
    allow_missing_judge_for_synthetic: bool = True

    # мягкий порог для judge_avg
    relaxed_low_judge_threshold: float = 2.0

    # too_short_text оставляем, если текст не совсем микроскопический
    allow_too_short_if_char_count_at_least: int = 180

    # prompt leak перепроверяем более строгими паттернами;
    # если не подтверждается — снимаем quarantine
    use_strict_prompt_leak_confirmation: bool = True

    # exact duplicates не карантиним, а дедуплицируем отдельно
    deduplicate_exact_texts: bool = True

    # если после снятия мягких флагов ничего не осталось — запись clean
    clean_if_only_soft_flags_resolved: bool = True

    # финальная политика: если остался только very-soft флаг, можно всё равно пустить в clean
    allow_remaining_soft_flags: bool = True


SOFT_METADATA_FLAGS = {
    "reported_char_count_mismatch",
    "reported_word_count_mismatch",
}

SOFT_QUALITY_FLAGS = {
    "missing_judge_avg_for_synthetic",
    "ai_fraction_mismatch",
}

TOPIC_FLAG_PREFIX = "topic_flag_"


In [9]:
# =========================
# CELL 3. Better prompt leak detection
# =========================

STRONG_PROMPT_LEAK_PATTERNS = [
    r"^\s*вот\s+переработан",                     # "Вот переработанный ..."
    r"^\s*вот\s+продолжени",                      # "Вот продолжение ..."
    r"^\s*ниже\s+приведен",                       # "Ниже приведен ..."
    r"^\s*конечно[,!\s]",                         # "Конечно, ..."
    r"^\s*output\s+only\b",
    r"^\s*translate\s+the\s+following\b",
    r"\byou\s+are\s+a\s+professional\s+translator\b",
    r"\bты\s+редактор\b",
    r"\bты\s+—\s+редактор\b",
    r"\bты\s+соавтор\b",
    r"\bтолько\s+текст\s+фрагмента\b",
    r"\bпродолжай\s+отсюда\b",
]

MEDIUM_PROMPT_LEAK_PATTERNS = [
    r"\btext:\b",
    r"\bfragment:\b",
    r"\bфрагмент:\b",
    r"\bтолько\s+продолжение\b",
    r"\bбез\s+пояснений\b",
]


def confirm_prompt_leak(text: str) -> Tuple[bool, List[str]]:
    """
    Более строгая проверка на утечку промпта.
    Старая эвристика была слишком широкая.
    Здесь мы требуем:
    - либо сильное совпадение,
    - либо несколько средних.
    """
    lower = text.lower()
    strong_hits = []
    medium_hits = []

    for pattern in STRONG_PROMPT_LEAK_PATTERNS:
        if re.search(pattern, lower):
            strong_hits.append(pattern)

    for pattern in MEDIUM_PROMPT_LEAK_PATTERNS:
        if re.search(pattern, lower):
            medium_hits.append(pattern)

    if len(strong_hits) >= 1:
        return True, strong_hits + medium_hits

    if len(medium_hits) >= 2:
        return True, medium_hits

    return False, strong_hits + medium_hits


In [10]:
# =========================
# CELL 4. Quarantine analytics
# =========================

def build_quarantine_issue_summary(records: List[Dict[str, Any]]) -> pd.DataFrame:
    """
    Считает, какие flags чаще всего встречаются в quarantine.
    """
    rows = []
    for r in records:
        if r["quality"]["status"] != "quarantine":
            continue
        flags = r["quality"].get("flags", [])
        if not flags:
            rows.append({"flag": "__no_flags__", "count": 1})
        else:
            for flag in flags:
                rows.append({"flag": flag, "count": 1})

    if not rows:
        return pd.DataFrame(columns=["flag", "count"])

    df = pd.DataFrame(rows)
    summary = df.groupby("flag", as_index=False)["count"].sum().sort_values("count", ascending=False)
    return summary


def build_strategy_issue_matrix(records: List[Dict[str, Any]]) -> pd.DataFrame:
    """
    Показывает, какие flags характерны для каких стратегий.
    """
    rows = []
    for r in records:
        strategy = r["meta"].get("strategy", "unknown")
        status = r["quality"]["status"]
        flags = r["quality"].get("flags", [])
        if not flags:
            rows.append({"strategy": strategy, "status": status, "flag": "__no_flags__"})
        else:
            for flag in flags:
                rows.append({"strategy": strategy, "status": status, "flag": flag})

    df = pd.DataFrame(rows)
    if df.empty:
        return df

    pivot = (
        df.groupby(["strategy", "flag"])
        .size()
        .reset_index(name="count")
        .sort_values(["strategy", "count"], ascending=[True, False])
    )
    return pivot


def sample_records_by_flag(
    records: List[Dict[str, Any]],
    flag: str,
    n: int = 5,
    random_state: int = 42
) -> pd.DataFrame:
    """
    Возвращает сэмплы записей с конкретным флагом для ручной проверки в ноутбуке.
    """
    matched = []
    for r in records:
        flags = r.get("quality", {}).get("flags", [])
        if flag in flags:
            matched.append({
                "id": r["id"],
                "group_id": r["group_id"],
                "label_name": r["meta"].get("dataset_label_name"),
                "strategy": r["meta"].get("strategy"),
                "judge_avg": r["meta"].get("judge_avg"),
                "char_count": r["meta"].get("char_count_actual"),
                "text_preview": r["text"][:500].replace("\n", " "),
                "flags": ", ".join(flags),
            })

    df = pd.DataFrame(matched)
    if df.empty:
        return df

    return df.sample(min(n, len(df)), random_state=random_state)


def basic_dataset_summary(records: List[Dict[str, Any]]) -> Dict[str, Any]:
    """
    Базовая сводка по набору records.
    """
    statuses = Counter(r["quality"]["status"] for r in records)
    labels = Counter(r["meta"].get("dataset_label_name") for r in records)
    strategies = Counter(r["meta"].get("strategy") for r in records)

    ai_fractions = [
        safe_float(r["meta"].get("ai_fraction_derived"))
        for r in records
        if safe_float(r["meta"].get("ai_fraction_derived")) is not None
    ]

    char_counts = [r["meta"].get("char_count_actual") for r in records if r["meta"].get("char_count_actual") is not None]
    word_counts = [r["meta"].get("word_count_actual") for r in records if r["meta"].get("word_count_actual") is not None]
    judge_avgs = [safe_float(r["meta"].get("judge_avg")) for r in records if safe_float(r["meta"].get("judge_avg")) is not None]

    total_chars = sum(r["meta"].get("char_count_actual", 0) for r in records)
    total_ai_chars = sum(r["meta"].get("ai_char_count", 0) for r in records)

    return {
        "n_records": len(records),
        "status_distribution": dict(statuses),
        "label_distribution": dict(labels),
        "strategy_distribution": dict(strategies),
        "char_count_stats": compute_numeric_stats(char_counts),
        "word_count_stats": compute_numeric_stats(word_counts),
        "ai_fraction_stats": compute_numeric_stats(ai_fractions),
        "judge_avg_stats": compute_numeric_stats(judge_avgs),
        "overall_ai_char_ratio": round(total_ai_chars / total_chars, 6) if total_chars > 0 else 0.0,
    }


In [11]:
# =========================
# CELL 5. Salvage logic
# =========================

def is_soft_flag(flag: str) -> bool:
    """
    Определяет, относится ли флаг к мягким/разрешимым.
    """
    if flag in SOFT_METADATA_FLAGS:
        return True
    if flag in SOFT_QUALITY_FLAGS:
        return True
    if flag.startswith(TOPIC_FLAG_PREFIX):
        return True
    if flag == "too_short_text":
        return True
    if flag == "exact_duplicate_text":
        return True
    if flag == "low_judge_avg":
        return True
    return False


def choose_best_duplicate_record(records: List[Dict[str, Any]]) -> Dict[str, Any]:
    """
    Выбирает лучший record среди exact duplicates.

    Приоритет:
    1. clean > quarantine
    2. выше judge_avg
    3. mixed > human > full_ai (для token-level mixed чуть полезнее)
    4. длиннее текст
    """
    label_priority = {
        "mixed": 3,
        "human": 2,
        "full_ai": 1,
    }

    def sort_key(r: Dict[str, Any]):
        status_score = 1 if r["quality"]["status"] == "clean" else 0
        judge_score = safe_float(r["meta"].get("judge_avg"))
        if judge_score is None:
            judge_score = -1.0
        label_score = label_priority.get(r["meta"].get("dataset_label_name"), 0)
        char_count = r["meta"].get("char_count_actual", 0)
        return (status_score, judge_score, label_score, char_count)

    best = sorted(records, key=sort_key, reverse=True)[0]
    return best


def review_record(record: Dict[str, Any], policy: ReviewPolicy) -> Dict[str, Any]:
    """
    Пересматривает статус записи.
    Снимает слишком жёсткие quarantine-флаги.
    """
    r = copy.deepcopy(record)

    original_flags = set(r.get("quality", {}).get("flags", []))
    remaining_flags = set(original_flags)
    resolved_flags = []
    review_notes = []

    # 1. metadata mismatch -> не причина quarantine
    if policy.ignore_metadata_mismatches:
        for flag in list(remaining_flags):
            if flag in SOFT_METADATA_FLAGS:
                remaining_flags.remove(flag)
                resolved_flags.append(flag)

    # 2. topic flags -> не карантинить для token-level обучения
    if policy.allow_topic_flags_for_training:
        for flag in list(remaining_flags):
            if flag.startswith(TOPIC_FLAG_PREFIX):
                remaining_flags.remove(flag)
                resolved_flags.append(flag)

    # 3. ai_fraction mismatch -> если spans валидны, считаем derived fraction главным
    if policy.allow_ai_fraction_mismatch and "ai_fraction_mismatch" in remaining_flags:
        remaining_flags.remove("ai_fraction_mismatch")
        resolved_flags.append("ai_fraction_mismatch")
        review_notes.append("use_derived_ai_fraction_instead_of_reported")

    # 4. missing judge avg -> не причина quarantine сама по себе
    if policy.allow_missing_judge_for_synthetic and "missing_judge_avg_for_synthetic" in remaining_flags:
        remaining_flags.remove("missing_judge_avg_for_synthetic")
        resolved_flags.append("missing_judge_avg_for_synthetic")

    # 5. low judge avg -> мягкий порог вместо жёсткого
    if "low_judge_avg" in remaining_flags:
        judge_avg = safe_float(r["meta"].get("judge_avg"))
        if judge_avg is not None and judge_avg >= policy.relaxed_low_judge_threshold:
            remaining_flags.remove("low_judge_avg")
            resolved_flags.append("low_judge_avg")
            review_notes.append(f"judge_avg_kept_with_relaxed_threshold={policy.relaxed_low_judge_threshold}")

    # 6. too short -> оставляем, если текст не совсем короткий
    if "too_short_text" in remaining_flags:
        char_count = r["meta"].get("char_count_actual", 0)
        if char_count >= policy.allow_too_short_if_char_count_at_least:
            remaining_flags.remove("too_short_text")
            resolved_flags.append("too_short_text")
            review_notes.append(f"kept_text_char_count={char_count}")

    # 7. prompt leak -> перепроверяем строгой эвристикой
    if "prompt_leak_pattern" in remaining_flags and policy.use_strict_prompt_leak_confirmation:
        confirmed, matched_patterns = confirm_prompt_leak(r["text"])
        if not confirmed:
            remaining_flags.remove("prompt_leak_pattern")
            resolved_flags.append("prompt_leak_pattern")
            review_notes.append("prompt_leak_not_confirmed_by_strict_rules")
        else:
            review_notes.append("prompt_leak_confirmed_by_strict_rules")
            review_notes.append(f"prompt_patterns={matched_patterns}")

    # exact duplicates решаем потом на этапе deduplication
    if "exact_duplicate_text" in remaining_flags and policy.deduplicate_exact_texts:
        remaining_flags.remove("exact_duplicate_text")
        resolved_flags.append("exact_duplicate_text")
        review_notes.append("will_be_handled_by_deduplication")

    # Определяем итоговый статус
    if len(remaining_flags) == 0:
        new_status = "clean"
    else:
        if policy.allow_remaining_soft_flags and all(is_soft_flag(f) for f in remaining_flags):
            new_status = "clean"
            review_notes.append("remaining_flags_are_soft_but_allowed")
        else:
            new_status = "quarantine"

    r["quality"]["status_original"] = r["quality"].get("status", "unknown")
    r["quality"]["status"] = new_status
    r["quality"]["flags_original"] = sorted(list(original_flags))
    r["quality"]["flags"] = sorted(list(remaining_flags))
    r["quality"]["resolved_flags"] = sorted(list(set(resolved_flags)))
    r["quality"]["review_notes"] = review_notes

    return r


def deduplicate_records(records: List[Dict[str, Any]]) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]]]:
    """
    Удаляет exact duplicates из clean records.
    Возвращает:
    - deduplicated records
    - duplicate report rows
    """
    by_hash = defaultdict(list)
    for r in records:
        text_hash = r["meta"].get("text_hash")
        by_hash[text_hash].append(r)

    deduped = []
    duplicate_report = []

    for text_hash, group in by_hash.items():
        if len(group) == 1:
            deduped.append(group[0])
            continue

        best = choose_best_duplicate_record(group)
        deduped.append(best)

        labels = sorted(set(r["meta"].get("dataset_label_name") for r in group))
        strategies = sorted(set(r["meta"].get("strategy") for r in group))
        ids = [r["id"] for r in group]

        duplicate_report.append({
            "text_hash": text_hash,
            "n_records": len(group),
            "kept_id": best["id"],
            "all_ids": ids,
            "labels_present": labels,
            "strategies_present": strategies,
            "has_label_conflict": len(labels) > 1,
        })

    return deduped, duplicate_report


In [12]:
# =========================
# CELL 6. Main review pipeline
# =========================

def run_extended_eda_and_salvage(
    input_base_dir: Path,
    output_base_dir: Path,
    val_size: float = 0.1,
    test_size: float = 0.1,
    seed: int = 42,
    policy: ReviewPolicy = ReviewPolicy(),
) -> Dict[str, Any]:
    """
    Загружает результаты первого этапа, делает углублённый EDA,
    пересматривает quarantine-логику и пересобирает итоговый датасет.
    """
    input_processed = input_base_dir / "processed"
    input_reports = input_base_dir / "reports"

    output_processed = output_base_dir / "processed"
    output_reports = output_base_dir / "reports"

    output_processed.mkdir(parents=True, exist_ok=True)
    output_reports.mkdir(parents=True, exist_ok=True)

    all_records_path = input_processed / "all_records.jsonl"
    if not all_records_path.exists():
        raise FileNotFoundError(f"Не найден файл: {all_records_path}")

    all_records = read_jsonl(all_records_path)

    # -------------------------
    # 1. Базовая сводка ДО пересмотра
    # -------------------------
    before_summary = basic_dataset_summary(all_records)

    quarantine_issue_summary_before = build_quarantine_issue_summary(all_records)
    quarantine_issue_summary_before.to_csv(
        output_reports / "quarantine_issue_summary_before.csv",
        index=False,
        encoding="utf-8"
    )

    strategy_issue_matrix_before = build_strategy_issue_matrix(all_records)
    strategy_issue_matrix_before.to_csv(
        output_reports / "strategy_issue_matrix_before.csv",
        index=False,
        encoding="utf-8"
    )

    # -------------------------
    # 2. Пересмотр статусов
    # -------------------------
    reviewed_records = [review_record(r, policy) for r in all_records]

    # -------------------------
    # 3. Базовая сводка ПОСЛЕ пересмотра
    # -------------------------
    after_review_summary = basic_dataset_summary(reviewed_records)

    quarantine_issue_summary_after = build_quarantine_issue_summary(reviewed_records)
    quarantine_issue_summary_after.to_csv(
        output_reports / "quarantine_issue_summary_after.csv",
        index=False,
        encoding="utf-8"
    )

    strategy_issue_matrix_after = build_strategy_issue_matrix(reviewed_records)
    strategy_issue_matrix_after.to_csv(
        output_reports / "strategy_issue_matrix_after.csv",
        index=False,
        encoding="utf-8"
    )

    # -------------------------
    # 4. Выделяем clean/quarantine после review
    # -------------------------
    clean_reviewed = [r for r in reviewed_records if r["quality"]["status"] == "clean"]
    quarantine_reviewed = [r for r in reviewed_records if r["quality"]["status"] == "quarantine"]

    # -------------------------
    # 5. Deduplication clean subset
    # -------------------------
    duplicate_report = []
    if policy.deduplicate_exact_texts:
        deduped_clean, duplicate_report = deduplicate_records(clean_reviewed)
    else:
        deduped_clean = clean_reviewed

    duplicate_report_columns = [
        "text_hash",
        "n_records",
        "kept_id",
        "all_ids",
        "labels_present",
        "strategies_present",
        "has_label_conflict",
    ]

    duplicate_report_df = pd.DataFrame(duplicate_report, columns=duplicate_report_columns)
    duplicate_report_df.to_csv(
        output_reports / "duplicate_report.csv",
        index=False,
        encoding="utf-8"
    )

    # -------------------------
    # 6. Group-aware split
    # -------------------------
    split_map = assign_group_splits(
        records=deduped_clean,
        val_size=val_size,
        test_size=test_size,
        seed=seed,
    )

    for r in deduped_clean:
        r["split"] = split_map[str(r["group_id"])]

    for r in quarantine_reviewed:
        r["split"] = None

    train_records = [r for r in deduped_clean if r["split"] == "train"]
    val_records = [r for r in deduped_clean if r["split"] == "val"]
    test_records = [r for r in deduped_clean if r["split"] == "test"]

    # -------------------------
    # 7. Отчёт по remaining quarantine
    # -------------------------
    remaining_quarantine_rows = []
    for r in quarantine_reviewed:
        remaining_quarantine_rows.append({
            "id": r["id"],
            "group_id": r["group_id"],
            "label_name": r["meta"].get("dataset_label_name"),
            "strategy": r["meta"].get("strategy"),
            "judge_avg": r["meta"].get("judge_avg"),
            "char_count": r["meta"].get("char_count_actual"),
            "remaining_flags": ", ".join(r["quality"].get("flags", [])),
            "resolved_flags": ", ".join(r["quality"].get("resolved_flags", [])),
            "review_notes": " | ".join(r["quality"].get("review_notes", [])),
            "text_preview": r["text"][:600].replace("\n", " "),
        })
    remaining_quarantine_df = pd.DataFrame(remaining_quarantine_rows)
    remaining_quarantine_df.to_csv(
        output_reports / "remaining_quarantine_samples.csv",
        index=False,
        encoding="utf-8"
    )

    # -------------------------
    # 8. Финальная summary
    # -------------------------
    def split_summary(records: List[Dict[str, Any]]) -> Dict[str, Any]:
        result = {}
        for split in ("train", "val", "test"):
            subset = [r for r in records if r.get("split") == split]
            result[split] = {
                "rows": len(subset),
                "unique_groups": len({str(r["group_id"]) for r in subset}),
                "label_distribution": dict(Counter(r["meta"].get("dataset_label_name") for r in subset)),
                "strategy_distribution": dict(Counter(r["meta"].get("strategy") for r in subset)),
            }
        return result

    final_summary = {
        "review_policy": asdict(policy),
        "before_review": before_summary,
        "after_review_before_dedup": after_review_summary,
        "after_review_after_dedup": basic_dataset_summary(deduped_clean),
        "n_remaining_quarantine": len(quarantine_reviewed),
        "n_duplicate_groups_removed_from_clean": int(len(duplicate_report_df)),
        "split_summary": split_summary(deduped_clean),
    }

    write_json(output_reports / "review_summary.json", final_summary)

    # -------------------------
    # 9. Экспорт файлов
    # -------------------------
    write_jsonl(output_processed / "reviewed_all_records.jsonl", reviewed_records)
    write_jsonl(output_processed / "reviewed_clean_records_before_dedup.jsonl", clean_reviewed)
    write_jsonl(output_processed / "reviewed_clean_records.jsonl", deduped_clean)
    write_jsonl(output_processed / "reviewed_quarantine_records.jsonl", quarantine_reviewed)
    write_jsonl(output_processed / "train.jsonl", train_records)
    write_jsonl(output_processed / "val.jsonl", val_records)
    write_jsonl(output_processed / "test.jsonl", test_records)

    # Дополнительная удобная таблица
    final_df_rows = []
    for r in reviewed_records:
        final_df_rows.append({
            "id": r["id"],
            "group_id": r["group_id"],
            "split": r.get("split"),
            "status": r["quality"]["status"],
            "label_name": r["meta"].get("dataset_label_name"),
            "strategy": r["meta"].get("strategy"),
            "judge_avg": r["meta"].get("judge_avg"),
            "char_count": r["meta"].get("char_count_actual"),
            "n_spans": len(r.get("spans", [])),
            "flags": ", ".join(r["quality"].get("flags", [])),
            "resolved_flags": ", ".join(r["quality"].get("resolved_flags", [])),
        })
    pd.DataFrame(final_df_rows).to_csv(
        output_reports / "final_records_overview.csv",
        index=False,
        encoding="utf-8"
    )

    print("=== REVIEW PIPELINE FINISHED ===")
    print(f"Input dir:  {input_base_dir}")
    print(f"Output dir: {output_base_dir}")
    print("\nBefore review:")
    print(json.dumps(before_summary, ensure_ascii=False, indent=2))
    print("\nAfter review + dedup:")
    print(json.dumps(final_summary["after_review_after_dedup"], ensure_ascii=False, indent=2))
    print("\nSplit summary:")
    print(json.dumps(final_summary["split_summary"], ensure_ascii=False, indent=2))

    return {
        "input_base_dir": input_base_dir,
        "output_base_dir": output_base_dir,
        "review_summary_path": output_reports / "review_summary.json",
        "remaining_quarantine_path": output_reports / "remaining_quarantine_samples.csv",
        "quarantine_issue_summary_before_path": output_reports / "quarantine_issue_summary_before.csv",
        "quarantine_issue_summary_after_path": output_reports / "quarantine_issue_summary_after.csv",
        "duplicate_report_path": output_reports / "duplicate_report.csv",
        "train_path": output_processed / "train.jsonl",
        "val_path": output_processed / "val.jsonl",
        "test_path": output_processed / "test.jsonl",
    }


In [13]:
# =========================
# CELL 7. Run
# =========================

policy = ReviewPolicy(
    ignore_metadata_mismatches=True,
    allow_topic_flags_for_training=True,
    allow_ai_fraction_mismatch=True,
    allow_missing_judge_for_synthetic=True,
    relaxed_low_judge_threshold=2.0,
    allow_too_short_if_char_count_at_least=180,
    use_strict_prompt_leak_confirmation=True,
    deduplicate_exact_texts=True,
    clean_if_only_soft_flags_resolved=True,
    allow_remaining_soft_flags=True,
)

artifacts = run_extended_eda_and_salvage(
    input_base_dir=Path("/token_detector_v1/"),
    output_base_dir=Path("/token_detector_v2_reviewed/"),
    val_size=0.1,
    test_size=0.1,
    seed=42,
    policy=policy,
)

artifacts


=== REVIEW PIPELINE FINISHED ===
Input dir:  /token_detector_v1
Output dir: /token_detector_v2_reviewed

Before review:
{
  "n_records": 3460,
  "status_distribution": {
    "quarantine": 2847,
    "clean": 613
  },
  "label_distribution": {
    "human": 825,
    "full_ai": 1489,
    "mixed": 1146
  },
  "strategy_distribution": {
    "original": 825,
    "paraphrase_deep": 703,
    "paraphrase_human_voice": 701,
    "partial_continuation": 567,
    "partial_middle_insert": 579,
    "back_translation": 85
  },
  "char_count_stats": {
    "count": 3460,
    "min": 779.0,
    "mean": 4447.335838,
    "median": 4068.0,
    "max": 37938.0,
    "std": 2575.144835
  },
  "word_count_stats": {
    "count": 3460,
    "min": 104.0,
    "mean": 581.371676,
    "median": 533.0,
    "max": 5425.0,
    "std": 351.430703
  },
  "ai_fraction_stats": {
    "count": 3460,
    "min": 0.0,
    "mean": 0.520791,
    "median": 0.341278,
    "max": 1.0,
    "std": 0.431541
  },
  "judge_avg_stats": {
    "c

{'input_base_dir': PosixPath('/token_detector_v1'),
 'output_base_dir': PosixPath('/token_detector_v2_reviewed'),
 'review_summary_path': PosixPath('/token_detector_v2_reviewed/reports/review_summary.json'),
 'remaining_quarantine_path': PosixPath('/token_detector_v2_reviewed/reports/remaining_quarantine_samples.csv'),
 'quarantine_issue_summary_before_path': PosixPath('/token_detector_v2_reviewed/reports/quarantine_issue_summary_before.csv'),
 'quarantine_issue_summary_after_path': PosixPath('/token_detector_v2_reviewed/reports/quarantine_issue_summary_after.csv'),
 'duplicate_report_path': PosixPath('/token_detector_v2_reviewed/reports/duplicate_report.csv'),
 'train_path': PosixPath('/token_detector_v2_reviewed/processed/train.jsonl'),
 'val_path': PosixPath('/token_detector_v2_reviewed/processed/val.jsonl'),
 'test_path': PosixPath('/token_detector_v2_reviewed/processed/test.jsonl')}

In [14]:
# =========================
# CELL 8. Quick inspection
# =========================

review_summary = json.loads(Path(artifacts["review_summary_path"]).read_text(encoding="utf-8"))
print(json.dumps(review_summary, ensure_ascii=False, indent=2))

print("\n--- TOP quarantine reasons BEFORE ---")
display(pd.read_csv(artifacts["quarantine_issue_summary_before_path"]).head(20))

print("\n--- TOP quarantine reasons AFTER ---")
display(pd.read_csv(artifacts["quarantine_issue_summary_after_path"]).head(20))

print("\n--- Duplicate report ---")
duplicate_report_df = pd.read_csv(artifacts["duplicate_report_path"])
display(duplicate_report_df.head(20))

print("\n--- Remaining quarantine samples ---")
remaining_quarantine_df = pd.read_csv(artifacts["remaining_quarantine_path"])
display(remaining_quarantine_df.head(20))


{
  "review_policy": {
    "ignore_metadata_mismatches": true,
    "allow_topic_flags_for_training": true,
    "allow_ai_fraction_mismatch": true,
    "allow_missing_judge_for_synthetic": true,
    "relaxed_low_judge_threshold": 2.0,
    "allow_too_short_if_char_count_at_least": 180,
    "use_strict_prompt_leak_confirmation": true,
    "deduplicate_exact_texts": true,
    "clean_if_only_soft_flags_resolved": true,
    "allow_remaining_soft_flags": true
  },
  "before_review": {
    "n_records": 3460,
    "status_distribution": {
      "quarantine": 2847,
      "clean": 613
    },
    "label_distribution": {
      "human": 825,
      "full_ai": 1489,
      "mixed": 1146
    },
    "strategy_distribution": {
      "original": 825,
      "paraphrase_deep": 703,
      "paraphrase_human_voice": 701,
      "partial_continuation": 567,
      "partial_middle_insert": 579,
      "back_translation": 85
    },
    "char_count_stats": {
      "count": 3460,
      "min": 779.0,
      "mean": 4447.3

,flag,count
0,topic_flag_politics,2086
1,topic_flag_religion,1577
2,topic_flag_philosophy,1171
3,low_judge_avg,116
4,ai_fraction_mismatch,3



--- TOP quarantine reasons AFTER ---


,flag,count



--- Duplicate report ---


,text_hash,n_records,kept_id,all_ids,labels_present,strategies_present,has_label_conflict



--- Remaining quarantine samples ---


EmptyDataError: No columns to parse from file

In [15]:
# =========================
# FIX CELL A. Make soft flags stricter
# =========================

def is_soft_flag(flag: str) -> bool:
    """
    Более строгая версия:
    low_judge_avg НЕ считаем автоматически мягким флагом.
    """
    if flag in SOFT_METADATA_FLAGS:
        return True
    if flag in SOFT_QUALITY_FLAGS:
        return True
    if flag.startswith(TOPIC_FLAG_PREFIX):
        return True
    if flag == "too_short_text":
        return True
    if flag == "exact_duplicate_text":
        return True
    # low_judge_avg ИСКЛЮЧАЕМ отсюда специально
    return False


In [16]:
# =========================
# FIX CELL B. Re-run with stricter review policy
# =========================

policy_v2 = ReviewPolicy(
    ignore_metadata_mismatches=True,
    allow_topic_flags_for_training=True,
    allow_ai_fraction_mismatch=True,
    allow_missing_judge_for_synthetic=True,
    relaxed_low_judge_threshold=2.0,
    allow_too_short_if_char_count_at_least=180,
    use_strict_prompt_leak_confirmation=True,
    deduplicate_exact_texts=True,
    clean_if_only_soft_flags_resolved=True,
    allow_remaining_soft_flags=False,   # ВАЖНО: выключаем
)

artifacts_v2 = run_extended_eda_and_salvage(
    input_base_dir=Path("/token_detector_v1/"),
    output_base_dir=Path("/token_detector_v3_reviewed_strict/"),
    val_size=0.1,
    test_size=0.1,
    seed=42,
    policy=policy_v2,
)

artifacts_v2


=== REVIEW PIPELINE FINISHED ===
Input dir:  /token_detector_v1
Output dir: /token_detector_v3_reviewed_strict

Before review:
{
  "n_records": 3460,
  "status_distribution": {
    "quarantine": 2847,
    "clean": 613
  },
  "label_distribution": {
    "human": 825,
    "full_ai": 1489,
    "mixed": 1146
  },
  "strategy_distribution": {
    "original": 825,
    "paraphrase_deep": 703,
    "paraphrase_human_voice": 701,
    "partial_continuation": 567,
    "partial_middle_insert": 579,
    "back_translation": 85
  },
  "char_count_stats": {
    "count": 3460,
    "min": 779.0,
    "mean": 4447.335838,
    "median": 4068.0,
    "max": 37938.0,
    "std": 2575.144835
  },
  "word_count_stats": {
    "count": 3460,
    "min": 104.0,
    "mean": 581.371676,
    "median": 533.0,
    "max": 5425.0,
    "std": 351.430703
  },
  "ai_fraction_stats": {
    "count": 3460,
    "min": 0.0,
    "mean": 0.520791,
    "median": 0.341278,
    "max": 1.0,
    "std": 0.431541
  },
  "judge_avg_stats": {

{'input_base_dir': PosixPath('/token_detector_v1'),
 'output_base_dir': PosixPath('/token_detector_v3_reviewed_strict'),
 'review_summary_path': PosixPath('/token_detector_v3_reviewed_strict/reports/review_summary.json'),
 'remaining_quarantine_path': PosixPath('/token_detector_v3_reviewed_strict/reports/remaining_quarantine_samples.csv'),
 'quarantine_issue_summary_before_path': PosixPath('/token_detector_v3_reviewed_strict/reports/quarantine_issue_summary_before.csv'),
 'quarantine_issue_summary_after_path': PosixPath('/token_detector_v3_reviewed_strict/reports/quarantine_issue_summary_after.csv'),
 'duplicate_report_path': PosixPath('/token_detector_v3_reviewed_strict/reports/duplicate_report.csv'),
 'train_path': PosixPath('/token_detector_v3_reviewed_strict/processed/train.jsonl'),
 'val_path': PosixPath('/token_detector_v3_reviewed_strict/processed/val.jsonl'),
 'test_path': PosixPath('/token_detector_v3_reviewed_strict/processed/test.jsonl')}

In [17]:
# =========================
# FIX CELL C. Safe reading helpers
# =========================

def safe_read_csv(path: Path) -> pd.DataFrame:
    """
    Безопасно читает CSV.
    Если файл пустой, возвращает пустой DataFrame.
    """
    if not path.exists():
        return pd.DataFrame()

    if path.stat().st_size == 0:
        return pd.DataFrame()

    try:
        return pd.read_csv(path)
    except pd.errors.EmptyDataError:
        return pd.DataFrame()


In [18]:
# =========================
# FIX CELL D. Inspect results safely
# =========================

review_summary_v2 = json.loads(Path(artifacts_v2["review_summary_path"]).read_text(encoding="utf-8"))
print(json.dumps(review_summary_v2, ensure_ascii=False, indent=2))

print("\n--- TOP quarantine reasons BEFORE ---")
display(safe_read_csv(artifacts_v2["quarantine_issue_summary_before_path"]).head(20))

print("\n--- TOP quarantine reasons AFTER ---")
display(safe_read_csv(artifacts_v2["quarantine_issue_summary_after_path"]).head(20))

print("\n--- Duplicate report ---")
duplicate_report_df_v2 = safe_read_csv(artifacts_v2["duplicate_report_path"])
display(duplicate_report_df_v2.head(20))

print("\n--- Remaining quarantine samples ---")
remaining_quarantine_df_v2 = safe_read_csv(artifacts_v2["remaining_quarantine_path"])
display(remaining_quarantine_df_v2.head(20))


{
  "review_policy": {
    "ignore_metadata_mismatches": true,
    "allow_topic_flags_for_training": true,
    "allow_ai_fraction_mismatch": true,
    "allow_missing_judge_for_synthetic": true,
    "relaxed_low_judge_threshold": 2.0,
    "allow_too_short_if_char_count_at_least": 180,
    "use_strict_prompt_leak_confirmation": true,
    "deduplicate_exact_texts": true,
    "clean_if_only_soft_flags_resolved": true,
    "allow_remaining_soft_flags": false
  },
  "before_review": {
    "n_records": 3460,
    "status_distribution": {
      "quarantine": 2847,
      "clean": 613
    },
    "label_distribution": {
      "human": 825,
      "full_ai": 1489,
      "mixed": 1146
    },
    "strategy_distribution": {
      "original": 825,
      "paraphrase_deep": 703,
      "paraphrase_human_voice": 701,
      "partial_continuation": 567,
      "partial_middle_insert": 579,
      "back_translation": 85
    },
    "char_count_stats": {
      "count": 3460,
      "min": 779.0,
      "mean": 4447.

,flag,count
0,topic_flag_politics,2086
1,topic_flag_religion,1577
2,topic_flag_philosophy,1171
3,low_judge_avg,116
4,ai_fraction_mismatch,3



--- TOP quarantine reasons AFTER ---


,flag,count
0,low_judge_avg,27



--- Duplicate report ---


,text_hash,n_records,kept_id,all_ids,labels_present,strategies_present,has_label_conflict



--- Remaining quarantine samples ---


,id,group_id,label_name,strategy,judge_avg,char_count,remaining_flags,resolved_flags,review_notes,text_preview
0,stanovlenie-religioznoy-identichnosti-lichnost...,stanovlenie-religioznoy-identichnosti-lichnost...,mixed,partial_continuation,1.5,5792,low_judge_avg,"topic_flag_philosophy, topic_flag_politics, to...",NaN,КРИТИКА И БИБЛИОГРАФИЯ УДК 159.923.2 ББК 86.21...
1,stanovlenie-religioznoy-identichnosti-lichnost...,stanovlenie-religioznoy-identichnosti-lichnost...,full_ai,paraphrase_human_voice,1.5,1963,low_judge_avg,"topic_flag_philosophy, topic_flag_religion",NaN,Религиозная идентичность как феномен личностно...
2,sovremennaya-sekulyarnaya-filosofiya-i-hristia...,sovremennaya-sekulyarnaya-filosofiya-i-hristia...,mixed,partial_continuation,1.5,5817,low_judge_avg,"topic_flag_philosophy, topic_flag_politics, to...",NaN,Иерей Максим Мищенко Смоленская православная д...
3,dinamicheskie-uprazhneniya-v-obuchenii-matemat...,dinamicheskie-uprazhneniya-v-obuchenii-matemat...,mixed,partial_middle_insert,1.0,3625,low_judge_avg,topic_flag_politics,NaN,Вестн. Моск. ун-та. Сер. 20. Педагогическое об...
4,a-l-bertie-delagard-chlen-tavricheskoy-uchenoy...,a-l-bertie-delagard-chlen-tavricheskoy-uchenoy...,mixed,partial_middle_insert,0.5,5462,low_judge_avg,"topic_flag_politics, topic_flag_religion",NaN,- [Симферополь]: Государственное издательство ...
5,realizm-kak-metodologicheskaya-osnova-nravstve...,realizm-kak-metodologicheskaya-osnova-nravstve...,mixed,partial_middle_insert,1.5,5304,low_judge_avg,"topic_flag_philosophy, topic_flag_religion",NaN,"Если молодые люди ищут духовности в религии, в..."
6,innovatsii-dlya-matematiki-perspektivnye-podho...,innovatsii-dlya-matematiki-perspektivnye-podho...,mixed,partial_middle_insert,1.5,7415,low_judge_avg,topic_flag_politics,NaN,"4. Структурная матрица торговли трёх стран S1,..."
7,smysl-istorii-v-religioznom-i-filosofskom-poni...,smysl-istorii-v-religioznom-i-filosofskom-poni...,mixed,partial_middle_insert,1.0,4176,low_judge_avg,"topic_flag_philosophy, topic_flag_politics, to...",NaN,Исторические науки Historical Sciences УДК 930...
8,my-byli-gotovy-prodolzhat-nashi-issledovaniya-...,my-byli-gotovy-prodolzhat-nashi-issledovaniya-...,mixed,partial_middle_insert,0.5,4780,low_judge_avg,"topic_flag_politics, topic_flag_religion",NaN,СВ вновь уловил вектор времени. В целом в слож...
9,otto-r-svyaschennoe-ob-irratsionalnom-v-idee-b...,otto-r-svyaschennoe-ob-irratsionalnom-v-idee-b...,full_ai,paraphrase_human_voice,1.5,5485,low_judge_avg,NaN,NaN,"Potter J., Wetherell M. in their work «Discour..."
